# Part 8 — Insert rows into Azure SQL and watch them mirror

This notebook is **Part 8** of the mirroring lab. The lab instructions are in `fabric-mirroring-azure-sql-lab.md` — complete Parts 3 to 7 there first, so your mirrored database exists and is running.

## Before you run: make sure the language is Python

This notebook is pure **Python** (pandas and pyodbc). It does not use Spark. Fabric notebooks often open as **PySpark**, so check before you run anything:

1. On the **Home** tab, open the language drop-down. It shows either **PySpark (Python)** or **Python**.
2. If it shows **PySpark (Python)**, select **Python** and confirm the switch.
3. Run the cells in order. The first session can take a few minutes to start.

## What this notebook does

1. Shows the newest rows in the source table **and** in the mirrored table, so you have a "before" picture of both sides.
2. Inserts 5 new rows into `SalesLT.SalesOrderDetail` in the source Azure SQL database.
3. Waits until those exact 5 rows appear in the mirrored database, then compares before and after.
4. Deletes those same 5 rows from the source and waits for the deletes to replicate.

You have your own dedicated database (`sqldb-di-workshop-<your user>`), so nothing you do here affects other participants.


## Cell 1 — Configuration

Replace `diuser1` in `SQL_DATABASE` with your own workshop user number (for example `sqldb-di-workshop-diuser2`). The server name comes from `workshop_environment.md`. If you gave the mirrored database a different name in Part 5, the cell finds it automatically when it is the only mirrored database in your workspace.


In [ ]:
import base64
import json

import requests
import notebookutils

SQL_SERVER   = "sqldiws2026.database.windows.net"   # from workshop_environment.md
SQL_DATABASE = "sqldb-di-workshop-diuser1"          # your per-user database — replace diuser1 with your workshop user
MIRRORED_DB  = "AdventureWorksLT_Mirrored"          # the name you gave the mirrored database in Part 5
TABLE        = "SalesLT.SalesOrderDetail"

# The account you are signed in as, read from your Microsoft Entra access token
fabric_token = notebookutils.credentials.getToken("pbi")
payload = fabric_token.split(".")[1]
claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
SIGNED_IN_AS = claims.get("upn") or claims.get("unique_name") or claims.get("preferred_username") or claims.get("email")

# Find the mirrored database in this workspace
WORKSPACE_ID   = notebookutils.runtime.context["currentWorkspaceId"]
WORKSPACE_NAME = notebookutils.runtime.context["currentWorkspaceName"]
resp = requests.get(f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/mirroredDatabases",
                    headers={"Authorization": f"Bearer {fabric_token}"})
resp.raise_for_status()
mirror_names = [m["displayName"] for m in resp.json().get("value", [])]
if MIRRORED_DB not in mirror_names:
    if len(mirror_names) == 1:
        print(f"'{MIRRORED_DB}' not found — using '{mirror_names[0]}', the only mirrored database in this workspace.")
        MIRRORED_DB = mirror_names[0]
    elif mirror_names:
        raise ValueError(f"Set MIRRORED_DB to one of these mirrored databases: {mirror_names}")
    else:
        raise ValueError("No mirrored database found in this workspace. Complete Parts 3 to 5 of the lab first.")

print("Signed in as    :", SIGNED_IN_AS)
print("Workspace       :", WORKSPACE_NAME)
print("Source          :", f"{SQL_SERVER} / {SQL_DATABASE}")
print("Mirrored item   :", MIRRORED_DB)
print("Exercise table  :", TABLE)


## Cell 2 — Connect to the source and look at the newest rows

Connects to Azure SQL Database as you, using your Microsoft Entra sign-in (no password), through the managed private endpoint in your workspace. It then shows the row count and the 5 newest rows **before** the insert, so you can compare later.

In [ ]:
import struct

import pandas as pd
import pyodbc

SQL_COPT_SS_ACCESS_TOKEN = 1256  # ODBC attribute for passing an Entra access token


def run_sql(sql):
    """Run T-SQL against the source Azure SQL database as you, and return the last result set."""
    token = notebookutils.credentials.getToken("https://database.windows.net/").encode("utf-16-le")
    conn_str = ("DRIVER={ODBC Driver 18 for SQL Server};"
                f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};Encrypt=yes;TrustServerCertificate=no;")
    with pyodbc.connect(conn_str, attrs_before={
            SQL_COPT_SS_ACCESS_TOKEN: struct.pack(f"<I{len(token)}s", len(token), token)}) as cn:
        cur = cn.cursor()
        cur.execute(sql)
        while cur.description is None and cur.nextset():
            pass
        df = pd.DataFrame()
        if cur.description:
            df = pd.DataFrame([tuple(r) for r in cur.fetchall()], columns=[d[0] for d in cur.description])
        cn.commit()
        return df


NEWEST_ROWS_SQL = f"""
SELECT TOP (5) SalesOrderDetailID, SalesOrderID, ProductID, OrderQty, UnitPrice, ModifiedDate
FROM {TABLE}
ORDER BY SalesOrderDetailID DESC
"""
COUNT_SQL = f"SELECT COUNT(*) AS row_count, MAX(SalesOrderDetailID) AS max_id FROM {TABLE}"

print("Connected to", SQL_SERVER, "as", run_sql("SELECT SUSER_NAME() AS login").iloc[0, 0])
source_before = run_sql(COUNT_SQL).iloc[0]
print(f"Source rows before the insert: {source_before.row_count} (highest SalesOrderDetailID: {source_before.max_id})")
print("Newest 5 rows in the source:")
display(run_sql(NEWEST_ROWS_SQL))

## Cell 3 — Look at the same rows in the mirrored database

Queries the mirrored copy of the table through the mirrored database's SQL analytics endpoint. Before the insert, the newest rows here should match the source rows from Cell 2.

In [ ]:
def run_mirror_sql(sql):
    """Run a read-only query against the mirrored database's SQL analytics endpoint."""
    with notebookutils.data.connect_to_artifact(MIRRORED_DB, WORKSPACE_ID, "MirroredDatabase") as conn:
        df = conn.query(sql)
    return df if df is not None else pd.DataFrame()  # query() returns None when no rows match


mirror_before = run_mirror_sql(COUNT_SQL).iloc[0]
print(f"Mirrored rows before the insert: {mirror_before.row_count} (highest SalesOrderDetailID: {mirror_before.max_id})")
print("Newest 5 rows in the mirror:")
display(run_mirror_sql(NEWEST_ROWS_SQL))

## Cell 4 — Insert 5 rows into the source

Picks a random existing sales order and 5 random products, inserts 5 `SalesOrderDetail` rows, and keeps the exact `SalesOrderDetailID` values it inserted in `exercise_ids`.

In [ ]:
INSERT_SQL = """
SET NOCOUNT ON;

DECLARE @InsertedRows TABLE (SalesOrderDetailID int PRIMARY KEY);

DECLARE @OrderID int = (
    SELECT TOP (1) SalesOrderID
    FROM SalesLT.SalesOrderHeader
    ORDER BY NEWID()
);

;WITH CandidateProducts AS (
    SELECT TOP (5) ProductID, ListPrice
    FROM SalesLT.Product
    WHERE ListPrice > 0
    ORDER BY NEWID()
)
INSERT INTO SalesLT.SalesOrderDetail
    (SalesOrderID, OrderQty, ProductID, UnitPrice, UnitPriceDiscount)
OUTPUT inserted.SalesOrderDetailID INTO @InsertedRows(SalesOrderDetailID)
SELECT
    @OrderID,
    CAST(1 + ABS(CHECKSUM(NEWID())) % 5 AS smallint) AS OrderQty,
    ProductID,
    ListPrice,
    0.00
FROM CandidateProducts;

SELECT d.SalesOrderDetailID, d.SalesOrderID, d.ProductID, d.OrderQty, d.UnitPrice, d.ModifiedDate
FROM SalesLT.SalesOrderDetail AS d
WHERE d.SalesOrderDetailID IN (SELECT SalesOrderDetailID FROM @InsertedRows)
ORDER BY d.SalesOrderDetailID;
"""

source_inserted = run_sql(INSERT_SQL)
exercise_ids = [int(v) for v in source_inserted["SalesOrderDetailID"]]
assert len(exercise_ids) == 5, "Expected exactly 5 inserted rows."

print(f"Inserted {len(exercise_ids)} rows into the source. Your SalesOrderDetailID values: {exercise_ids}")
display(source_inserted)

## Cell 5 — Wait for your rows to appear in the mirror

Checks the mirrored database every 15 seconds, for up to 5 minutes, until all 5 of your rows are there. Replication usually takes about a minute. If the cell finishes before all rows arrive, run it again.

In [ ]:
import time

ids_sql = ",".join(str(i) for i in exercise_ids)
deadline = time.time() + 300
while True:
    mirrored_rows = run_mirror_sql(f"""
        SELECT SalesOrderDetailID, SalesOrderID, ProductID, OrderQty, UnitPrice, ModifiedDate
        FROM {TABLE}
        WHERE SalesOrderDetailID IN ({ids_sql})
        ORDER BY SalesOrderDetailID""")
    print(f"{time.strftime('%H:%M:%S')}  {len(mirrored_rows)} of {len(exercise_ids)} of your rows are in the mirror")
    if len(mirrored_rows) == len(exercise_ids) or time.time() > deadline:
        break
    time.sleep(15)

if len(mirrored_rows) == len(exercise_ids):
    print("\nAll of your rows replicated to the mirrored database:")
else:
    print("\nNot all rows have replicated yet. Check Monitor replication (below), then run this cell again.")
display(mirrored_rows)

source_after = run_sql(COUNT_SQL).iloc[0]
mirror_after = run_mirror_sql(COUNT_SQL).iloc[0]
display(pd.DataFrame({
    "Rows before insert": [source_before.row_count, mirror_before.row_count],
    "Rows now": [source_after.row_count, mirror_after.row_count],
    "Highest SalesOrderDetailID now": [source_after.max_id, mirror_after.max_id],
}, index=["Source (Azure SQL)", "Mirror (Fabric)"]))

## Cell 6 — Delete the rows you inserted

Deletes the same 5 rows from the source database, so mirroring has a delete to replicate as well as an insert.


In [ ]:
DELETE_SQL = f"""
DELETE FROM {TABLE}
WHERE SalesOrderDetailID IN ({ids_sql});

SELECT COUNT(*) AS row_count, MAX(SalesOrderDetailID) AS max_id FROM {TABLE};
"""

source_deleted = run_sql(DELETE_SQL).iloc[0]
print(f"Deleted your {len(exercise_ids)} rows from the source.")
print(f"Source rows after the delete: {source_deleted.row_count} (highest SalesOrderDetailID: {source_deleted.max_id})")


## Cell 7 — Wait for the deletes to replicate

Checks the mirrored database every 15 seconds, for up to 5 minutes, until your 5 rows are gone from the mirror.


In [ ]:
deadline = time.time() + 300
while True:
    still_there = run_mirror_sql(f"""
        SELECT SalesOrderDetailID
        FROM {TABLE}
        WHERE SalesOrderDetailID IN ({ids_sql})""")
    print(f"{time.strftime('%H:%M:%S')}  {len(still_there)} of your {len(exercise_ids)} rows are still in the mirror")
    if len(still_there) == 0 or time.time() > deadline:
        break
    time.sleep(15)

if len(still_there) == 0:
    print("\nAll of your rows have been removed from the mirrored database.")
else:
    print("\nSome rows have not replicated the delete yet. Check Monitor replication, then run this cell again.")

source_final = run_sql(COUNT_SQL).iloc[0]
mirror_final = run_mirror_sql(COUNT_SQL).iloc[0]
display(pd.DataFrame({
    "Rows before insert": [source_before.row_count, mirror_before.row_count],
    "Rows after insert": [source_after.row_count, mirror_after.row_count],
    "Rows after delete": [source_final.row_count, mirror_final.row_count],
}, index=["Source (Azure SQL)", "Mirror (Fabric)"]))


## Cell 8 — Monitor replication in the Fabric portal

You can also watch replication from the mirrored database item:

1. Open your mirrored database item (`AdventureWorksLT_Mirrored`) in your workspace.
2. Select **Monitor replication** and confirm the status is **Running**.
3. Find `SalesLT.SalesOrderDetail`. Its **Rows replicated** and **Last refresh** values update after your insert and delete.

## If something goes wrong

| Symptom | What to check |
| --- | --- |
| `NameError: name 'spark' is not defined` or other Spark errors | This notebook doesn't use Spark. If you see Spark errors, you're running an older copy of the notebook. Import this one again. |
| Cell 2 times out or fails to log in | In **Workspace settings → Network security**, the managed private endpoint to `sqldiws2026` must be **Approved**, and your account must be in the workshop participants group. |
| `Cannot open database "sqldb-di-workshop-diuser1"` | You did not replace `diuser1` in Cell 1 with your own workshop user number. |
| Cell 3 or Cell 5 can't find the mirrored database | Complete Parts 3 to 5 of the lab, and set `MIRRORED_DB` in Cell 1 to the name you used. |
| Your rows don't appear in or disappear from the mirror | Check **Monitor replication** and confirm the `fabriccapdiws` capacity is running, then run the waiting cell again. |
| `LineTotal` isn't in the mirrored table | It's a non-persisted computed column in AdventureWorksLT, so mirroring doesn't replicate it. Compute it in Fabric if you need it. |
